In [1]:
# ============================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================================

# Import standard operating system interface module for file and directory handling
import os
# Import random module for setting random seeds and randomized operations
import random
# Import NumPy library for numerical arrays, matrix math, and vector operations
import numpy as np
# Import pandas library for DataFrame handling and spreadsheet manipulation
import pandas as pd
# Import matplotlib pyplot for creating charts and visual plots
import matplotlib.pyplot as plt
import seaborn as sns

# Import PIL Image to open, resize, and convert image files
from PIL import Image

# Import PyTorch core deep learning framework
import torch
# Import PyTorch core deep learning framework
import torch.nn as nn
# Import Dataset base class and DataLoader for batching and loading data
from torch.utils.data import Dataset, DataLoader

# Import torchvision package for computer vision models and utilities
import torchvision
from torchvision import transforms, models

# Import dataset splitting utilities from scikit-learn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
# Import evaluation metrics (accuracy, confusion matrix, precision, recall, F1, ROC-AUC)
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report)

# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------

# Define fixed random seed value to ensure reproducible results across runs
SEED = 42

# Seed Python built-in random number generator
random.seed(SEED)
# Seed NumPy random number generator
np.random.seed(SEED)
# Seed PyTorch CPU random number generator
torch.manual_seed(SEED)

# Check if a CUDA-capable GPU is available
if torch.cuda.is_available():
    # Seed all GPU CUDA random number generators for reproducibility
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

# Select GPU device ('cuda') if available, otherwise fallback to CPU ('cpu')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch Version :", torch.__version__)
print("Device          :", device)

# Check if a CUDA-capable GPU is available
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

PyTorch Version : 2.8.0+cpu
Device          : cpu
Using CPU


In [2]:
# ============================================================
# CELL 2 — DATASET PATHS
# ============================================================

# Define base project directory path containing datasets and assets
path2data = r"F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/"

# Define directory path containing raw training image files
path2train = os.path.join(path2data,"training_drion")

# Construct full file path to the Excel spreadsheet containing labels
path2labels = os.path.join(path2data,"drion_training.xlsx")

print("Image directory :", path2train)
print("Excel file      :", path2labels)

print("Image directory exists :", os.path.exists(path2train))
print("Excel file exists     :", os.path.exists(path2labels))

Image directory : F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/training_drion
Excel file      : F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/drion_training.xlsx
Image directory exists : True
Excel file exists     : True


In [3]:
# ============================================================
#  READ LABEL FILE
# ============================================================

# Read the Excel spreadsheet into a pandas DataFrame
labels_df = pd.read_excel(path2labels)

print("Dataset shape:", labels_df.shape)

print("\nColumns:")
print(labels_df.columns.tolist())

print("\nFirst 5 rows:")
display(labels_df.head())

print("\nLast 5 rows:")
display(labels_df.tail())

print("\nClass distribution:")
print(labels_df["label"].value_counts())

Dataset shape: (86, 3)

Columns:
['ID', 'imgName', 'label']

First 5 rows:


,ID,imgName,label
0,1,image_001.jpg,0
1,2,image_002.jpg,0
2,3,image_003.jpg,0
3,4,image_004.jpg,0
4,5,image_005.jpg,0



Last 5 rows:


,ID,imgName,label
81,82,image_091.jpg,0
82,83,image_092.jpg,0
83,84,image_093.jpg,1
84,85,image_097.jpg,1
85,86,image_098.jpg,1



Class distribution:
label
0    47
1    39
Name: count, dtype: int64


In [4]:
# ============================================================
# CREATE IMAGE PATHS
# ============================================================

labels_df["image_path"] = labels_df["imgName"].apply(lambda x: os.path.join(path2train, str(x)))

# Check files
labels_df["file_exists"] = labels_df["image_path"].apply(os.path.exists)

print("Total records :", len(labels_df))
print("Valid images  :", labels_df["file_exists"].sum())
print("Missing images:", (~labels_df["file_exists"]).sum())

# Show missing images
missing_df = labels_df[~labels_df["file_exists"]]

if len(missing_df) > 0:
    print("\nMissing image files:")
    display(
        missing_df[
            ["imgName", "label", "image_path"]
        ]
    )

# Keep only valid images
labels_df = labels_df[labels_df["file_exists"]].reset_index(drop=True)

print("\nDataset after removing missing files:")
print(labels_df.shape)

Total records : 86
Valid images  : 86
Missing images: 0

Dataset after removing missing files:
(86, 5)


In [5]:
# ============================================================
#  ENCODE LABELS
# ============================================================

print("Original labels:")
print(labels_df["label"].unique())

# ------------------------------------------------------------
#  TEXT LABELS
# ------------------------------------------------------------

if labels_df["label"].dtype == object:

    label_strings = (labels_df["label"].astype(str).str.strip().str.lower())

    # Extract distinct unique class label values from the dataset
    unique_labels = sorted(label_strings.unique())

    print("\nDetected labels:", unique_labels)

    # Explicit glaucoma mapping
    if "normal" in unique_labels and "glaucoma" in unique_labels:

        label_mapping = {"normal": 0,"glaucoma": 1}

    else:
        # Generic binary mapping
        label_mapping = {
            label: idx
            for idx, label in enumerate(unique_labels)
        }

    labels_df["encoded_label"] = label_strings.map(
        label_mapping
    )

else:

    # Numeric labels
    labels_df["encoded_label"] = (
        pd.to_numeric(labels_df["label"])
        .astype(int)
    )

    label_mapping = {
        0: 0,
        1: 1
    }

print("\nLabel mapping:")
print(label_mapping)

print("\nEncoded class distribution:")
print(
    labels_df["encoded_label"]
    .value_counts()
    .sort_index()
)

Original labels:
[0 1]

Label mapping:
{0: 0, 1: 1}

Encoded class distribution:
encoded_label
0    47
1    39
Name: count, dtype: int64


In [6]:
# ============================================================
#  PREPARE X AND y
# ============================================================

image_paths = labels_df["image_path"].values

labels = labels_df["encoded_label"].values.astype(np.int64)

print("Number of images:", len(image_paths))
print("Number of labels:", len(labels))

print("\nClass counts:")

unique, counts = np.unique(labels, return_counts=True)

for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

Number of images: 86
Number of labels: 86

Class counts:
Class 0: 47
Class 1: 39


In [7]:
# ============================================================
#  TRAIN / TEST SPLIT
# ============================================================

# Split dataset into stratified 80% train and 20% test partitions
train_paths, test_paths, train_labels, test_labels = train_test_split(image_paths,labels,test_size=0.20,random_state=SEED,stratify=labels)

print("Training samples:", len(train_paths))
print("Testing samples :", len(test_paths))

print("\nTraining distribution:")
print(
    pd.Series(train_labels)
    .value_counts()
    .sort_index()
)

print("\nTesting distribution:")
print(
    pd.Series(test_labels)
    .value_counts()
    .sort_index()
)

Training samples: 68
Testing samples : 18

Training distribution:
0    37
1    31
Name: count, dtype: int64

Testing distribution:
0    10
1     8
Name: count, dtype: int64


In [8]:
# ============================================================
#  IMAGE TRANSFORMS
# ============================================================

IMG_SIZE = 224

# Define training image augmentation and normalization pipeline
train_transform = transforms.Compose([
    # Resize image dimensions to uniform spatial resolution
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Randomly flip image horizontally for data augmentation
    transforms.RandomHorizontalFlip(),

    # Randomly rotate image slightly to improve model generalization
    transforms.RandomRotation(10),

    # Convert PIL Image to PyTorch FloatTensor with pixels scaled to [0.0, 1.0]
    transforms.ToTensor(),

    # Normalize image tensor using ImageNet standard mean and standard deviation
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([
    # Resize image dimensions to uniform spatial resolution
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Convert PIL Image to PyTorch FloatTensor with pixels scaled to [0.0, 1.0]
    transforms.ToTensor(),

    # Normalize image tensor using ImageNet standard mean and standard deviation
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
# ============================================================
#  DATASET CLASS
# ============================================================

class DRIONDataset(Dataset):

    def __init__(
        self,
        image_paths,
        labels,
        transform=None
    ):

        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    # Return total number of samples present in the dataset
    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]

        # Open image file from disk and convert to 3-channel RGB color format
        image = Image.open(
            image_path
        ).convert("RGB")

        label = int(self.labels[idx])

        # Apply transformation pipeline if specified
        if self.transform is not None:

            # Transform raw image into normalized PyTorch tensor
            image = self.transform(image)

        # Return processed image tensor and corresponding class label
        return image, label

In [10]:
# ============================================================
#  CNN MODEL
# ============================================================

# Function to construct ResNet-18 architecture with pretrained weights
def create_resnet18():

    # Load pretrained ResNet-18 model architecture
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Retrieve number of input features entering the final classification layer
    num_features = model.fc.in_features

    # Replace final fully connected layer with a Linear layer outputting 2 classes
    model.fc = nn.Linear(num_features,1)

    # Return the configured neural network model
    return model


# Function to construct ResNet-34 architecture with pretrained weights
def create_resnet34():

    # Load pretrained ResNet-34 model architecture
    model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

    # Retrieve number of input features entering the final classification layer
    num_features = model.fc.in_features

    # Replace final fully connected layer with a Linear layer outputting 2 classes
    model.fc = nn.Linear(num_features,1)

    # Return the configured neural network model
    return model


# Function to construct DenseNet-121 architecture with pretrained weights
def create_densenet121():

    # Load pretrained DenseNet-121 model architecture
    model = models.densenet121(
        weights=models.DenseNet121_Weights.DEFAULT
    )

    # Retrieve number of input features entering the DenseNet classifier layer
    num_features = model.classifier.in_features

    # Replace final DenseNet classifier with a Linear layer outputting 2 classes
    model.classifier = nn.Linear(num_features,1)

    # Return the configured neural network model
    return model

In [11]:
# ============================================================
# CELL 11 — TRAINING FUNCTION
# ============================================================

def train_model(model,train_loader,val_loader,epochs=10,lr=1e-4):

    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    # Initialize Adam optimizer with model weights and learning rate
    optimizer = torch.optim.Adam(model.parameters(),lr=lr)

    best_val_loss = float("inf")

    best_state = None

    for epoch in range(epochs):

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        # Put neural network model in training mode (enables dropout and batchnorm updates)
        model.train()

        running_loss = 0.0

        for images, labels_batch in train_loader:

            images = images.to(device)

            labels_batch = (labels_batch.float().to(device))

            # Clear previously accumulated parameter gradients before current backward pass
            optimizer.zero_grad()

            # Execute forward pass: feed batch images into the neural network to get logits
            outputs = model(images).squeeze(1)

            loss = criterion(outputs,labels_batch)
            # Execute backward pass: compute gradients of loss with respect to all model parameters
            loss.backward()
            # Update neural network weights using computed gradients
            optimizer.step()
            running_loss += (loss.item()* images.size(0))

        train_loss = (running_loss/ len(train_loader.dataset))

        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        # Put neural network model in evaluation mode (disables dropout and fixes batchnorm)
        model.eval()

        val_running_loss = 0.0

        # Disable gradient calculations to accelerate inference and conserve GPU memory
        with torch.no_grad():

            for images, labels_batch in val_loader:

                images = images.to(device)

                labels_batch = (labels_batch.float().to(device))

                # Execute forward pass: feed batch images into the neural network to get logits
                outputs = model(images).squeeze(1)

                loss = criterion(outputs,labels_batch)

                val_running_loss += (loss.item()* images.size(0))

        val_loss = (val_running_loss / len(val_loader.dataset))

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )

        # ----------------------------------------------------
        # SAVE BEST MODEL
        # ----------------------------------------------------

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }

    # Restore best model
    if best_state is not None:

        model.load_state_dict(best_state)

    # Return the configured neural network model
    return model

In [12]:
# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_model(model,data_loader):

    # Put neural network model in evaluation mode (disables dropout and fixes batchnorm)
    model.eval()

    probabilities = []

    true_labels = []

    # Disable gradient calculations to accelerate inference and conserve GPU memory
    with torch.no_grad():

        for images, labels_batch in data_loader:

            images = images.to(device)

            # Execute forward pass: feed batch images into the neural network to get logits
            outputs = model(images).squeeze(1)

            probs = torch.sigmoid(outputs)

            probabilities.extend(probs.cpu().numpy())

            true_labels.extend(labels_batch.numpy())

    return (
        np.array(probabilities),
        np.array(true_labels)
    )

In [13]:
# ============================================================
#  FOLD OOF STACKING (out-of-fold predictions)
# ============================================================

N_SPLITS = 5
BATCH_SIZE = 8
EPOCHS = 10
LEARNING_RATE = 1e-4

skf = StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)

# ------------------------------------------------------------
# OOF prediction matrix
#
# Column 0 = ResNet18
# Column 1 = ResNet34
# Column 2 = DenseNet121
# ------------------------------------------------------------

oof_predictions = np.zeros((len(train_paths), 3))

fold_models = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(train_paths, train_labels)
):

    print("\n")
    print("=" * 70)
    print(f"FOLD {fold + 1}/{N_SPLITS}")
    print("=" * 70)

    # --------------------------------------------------------
    # Fold data
    # --------------------------------------------------------

    fold_train_paths = train_paths[train_idx]
    fold_train_labels = train_labels[train_idx]

    fold_val_paths = train_paths[val_idx]
    fold_val_labels = train_labels[val_idx]

    # --------------------------------------------------------
    # Datasets
    # --------------------------------------------------------

    fold_train_dataset = DRIONDataset(fold_train_paths,fold_train_labels,transform=train_transform)

    fold_val_dataset = DRIONDataset(fold_val_paths,fold_val_labels,transform=val_transform)

    fold_train_loader = DataLoader(
        fold_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    fold_val_loader = DataLoader(
        fold_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    # --------------------------------------------------------
    # Create three CNN models
    # --------------------------------------------------------

    model1 = create_resnet18()

    model2 = create_resnet34()

    model3 = create_densenet121()

    # --------------------------------------------------------
    # MODEL 1 — ResNet18
    # --------------------------------------------------------

    print("\nTraining ResNet18...")

    model1 = train_model(model1,fold_train_loader,fold_val_loader,epochs=EPOCHS,lr=LEARNING_RATE)

    pred1, _ = predict_model(model1,fold_val_loader)

    # --------------------------------------------------------
    # MODEL 2 — ResNet34
    # --------------------------------------------------------

    print("\nTraining ResNet34...")

    model2 = train_model(model2,fold_train_loader,fold_val_loader,epochs=EPOCHS,lr=LEARNING_RATE)

    pred2, _ = predict_model(model2,fold_val_loader)

    # --------------------------------------------------------
    # MODEL 3 — DenseNet121
    # --------------------------------------------------------

    print("\nTraining DenseNet121...")

    model3 = train_model(model3,fold_train_loader,fold_val_loader,epochs=EPOCHS,lr=LEARNING_RATE)

    pred3, _ = predict_model(model3,fold_val_loader)

    # --------------------------------------------------------
    # Store OOF predictions
    # --------------------------------------------------------

    oof_predictions[val_idx,0] = pred1

    oof_predictions[val_idx,1] = pred2

    oof_predictions[val_idx,2] = pred3

    # --------------------------------------------------------
    # Store models
    # --------------------------------------------------------

    fold_models.append([model1, model2, model3])

    print(
        f"\nFold {fold + 1} completed."
    )



FOLD 1/5

Training ResNet18...
Epoch [1/10] Train Loss: 0.7039 | Val Loss: 0.7181
Epoch [2/10] Train Loss: 0.5794 | Val Loss: 0.8083
Epoch [3/10] Train Loss: 0.4943 | Val Loss: 0.8559
Epoch [4/10] Train Loss: 0.3345 | Val Loss: 1.1861
Epoch [5/10] Train Loss: 0.2317 | Val Loss: 1.3559
Epoch [6/10] Train Loss: 0.2411 | Val Loss: 1.3438
Epoch [7/10] Train Loss: 0.2949 | Val Loss: 1.4644
Epoch [8/10] Train Loss: 0.0869 | Val Loss: 1.7072
Epoch [9/10] Train Loss: 0.1859 | Val Loss: 1.4482
Epoch [10/10] Train Loss: 0.0726 | Val Loss: 1.6346

Training ResNet34...
Epoch [1/10] Train Loss: 0.8375 | Val Loss: 0.7729
Epoch [2/10] Train Loss: 0.5978 | Val Loss: 0.8075
Epoch [3/10] Train Loss: 0.3475 | Val Loss: 1.0530
Epoch [4/10] Train Loss: 0.2879 | Val Loss: 0.9256
Epoch [5/10] Train Loss: 0.1841 | Val Loss: 1.0899
Epoch [6/10] Train Loss: 0.1293 | Val Loss: 1.1301
Epoch [7/10] Train Loss: 0.1589 | Val Loss: 1.1935
Epoch [8/10] Train Loss: 0.2125 | Val Loss: 1.6838
Epoch [9/10] Train Loss: 0

In [14]:
# ============================================================
#  META-LEARNER
# ============================================================

meta_learner = LogisticRegression(max_iter=1000,random_state=SEED)

meta_learner.fit(oof_predictions,train_labels)

print("Meta-learner trained successfully.")

print("\nMeta-learner coefficients:")

print(
    meta_learner.coef_
)

print("\nMeta-learner intercept:")

print(
    meta_learner.intercept_
)

Meta-learner trained successfully.

Meta-learner coefficients:
[[ 1.15346723  0.7914734  -0.04682642]]

Meta-learner intercept:
[-1.02782014]


In [15]:
# ============================================================
# CELL 14 — FINAL CNN TRAINING
# ============================================================
#
# IMPORTANT:
# The TEST set is NOT used during training.
#
# The three final CNNs are trained using ALL available
# training samples.
#
# The number of epochs is fixed using the previous
# cross-validation experiments.
# ============================================================

FINAL_EPOCHS = EPOCHS
FINAL_LR = LEARNING_RATE

# ------------------------------------------------------------
# FULL TRAINING DATA
# ------------------------------------------------------------

full_train_dataset = DRIONDataset(train_paths,train_labels,transform=train_transform)

full_train_loader = DataLoader(full_train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)

# ------------------------------------------------------------
# TEST DATA
# ------------------------------------------------------------

test_dataset = DRIONDataset(
    test_paths,
    test_labels,
    transform=val_transform
)

# Create DataLoader for test set to handle batching
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("=" * 70)
print("FINAL CNN TRAINING")
print("=" * 70)

print("Number of training samples:", len(full_train_dataset))
print("Number of test samples    :", len(test_dataset))
print("Final epochs              :", FINAL_EPOCHS)
print("Learning rate             :", FINAL_LR)

FINAL CNN TRAINING
Number of training samples: 68
Number of test samples    : 18
Final epochs              : 10
Learning rate             : 0.0001


In [16]:
# ============================================================
# FINAL TRAINING FUNCTION
# ============================================================

def train_final_model(model,train_loader,epochs=10,lr=1e-4,model_name="Model"):

    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    # Initialize Adam optimizer with model weights and learning rate
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    print("\n" + "=" * 70)
    print(f"TRAINING FINAL {model_name}")
    print("=" * 70)

    for epoch in range(epochs):

        # Put neural network model in training mode (enables dropout and batchnorm updates)
        model.train()

        running_loss = 0.0

        for images, labels_batch in train_loader:

            images = images.to(device)

            labels_batch = (
                labels_batch
                .float()
                .to(device)
            )

            # Clear previously accumulated parameter gradients before current backward pass
            optimizer.zero_grad()

            # Execute forward pass: feed batch images into the neural network to get logits
            outputs = model(images).squeeze(1)

            loss = criterion(
                outputs,
                labels_batch
            )

            # Execute backward pass: compute gradients of loss with respect to all model parameters
            loss.backward()

            # Update neural network weights using computed gradients
            optimizer.step()

            running_loss += (
                loss.item() *
                images.size(0)
            )

        epoch_loss = (
            running_loss /
            len(train_loader.dataset)
        )

        print(
            f"{model_name} | "
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {epoch_loss:.4f}"
        )

    # Return the configured neural network model
    return model

In [17]:
print("\nTraining final ResNet18...")

final_resnet18 = create_resnet18()

final_resnet18 = train_final_model(model=final_resnet18,train_loader=full_train_loader,epochs=FINAL_EPOCHS,lr=FINAL_LR,model_name="ResNet18")

print("\nFinal ResNet18 training completed.")


Training final ResNet18...

TRAINING FINAL ResNet18
ResNet18 | Epoch [1/10] | Train Loss: 0.7488
ResNet18 | Epoch [2/10] | Train Loss: 0.5550
ResNet18 | Epoch [3/10] | Train Loss: 0.4890
ResNet18 | Epoch [4/10] | Train Loss: 0.3831
ResNet18 | Epoch [5/10] | Train Loss: 0.2902
ResNet18 | Epoch [6/10] | Train Loss: 0.2353
ResNet18 | Epoch [7/10] | Train Loss: 0.1849
ResNet18 | Epoch [8/10] | Train Loss: 0.1614
ResNet18 | Epoch [9/10] | Train Loss: 0.2117
ResNet18 | Epoch [10/10] | Train Loss: 0.1879

Final ResNet18 training completed.


In [ ]:
# ============================================================
#  FINAL RESNET34
# ============================================================

print("\nTraining final ResNet34...")

final_resnet34 = create_resnet34()

final_resnet34 = train_final_model(
    model=final_resnet34,
    train_loader=full_train_loader,
    epochs=FINAL_EPOCHS,
    lr=FINAL_LR,
    model_name="ResNet34"
)

print("\nFinal ResNet34 training completed.")


Training final ResNet34...

TRAINING FINAL ResNet34
ResNet34 | Epoch [1/10] | Train Loss: 0.7461
ResNet34 | Epoch [2/10] | Train Loss: 0.6746
ResNet34 | Epoch [3/10] | Train Loss: 0.5783
ResNet34 | Epoch [4/10] | Train Loss: 0.4060
ResNet34 | Epoch [5/10] | Train Loss: 0.3605
ResNet34 | Epoch [6/10] | Train Loss: 0.2915
ResNet34 | Epoch [7/10] | Train Loss: 0.2419
ResNet34 | Epoch [8/10] | Train Loss: 0.2487
ResNet34 | Epoch [9/10] | Train Loss: 0.2389
ResNet34 | Epoch [10/10] | Train Loss: 0.2059

Final ResNet34 training completed.


In [ ]:
# ============================================================
#  FINAL DENSENET121
# ============================================================

print("\nTraining final DenseNet121...")

final_densenet121 = create_densenet121()

final_densenet121 = train_final_model(
    model=final_densenet121,
    train_loader=full_train_loader,
    epochs=FINAL_EPOCHS,
    lr=FINAL_LR,
    model_name="DenseNet121"
)

print("\nFinal DenseNet121 training completed.")


Training final DenseNet121...

TRAINING FINAL DenseNet121
DenseNet121 | Epoch [1/10] | Train Loss: 0.7016
DenseNet121 | Epoch [2/10] | Train Loss: 0.6256
DenseNet121 | Epoch [3/10] | Train Loss: 0.6166
DenseNet121 | Epoch [4/10] | Train Loss: 0.5295
DenseNet121 | Epoch [5/10] | Train Loss: 0.4781
DenseNet121 | Epoch [6/10] | Train Loss: 0.3858
DenseNet121 | Epoch [7/10] | Train Loss: 0.3195
DenseNet121 | Epoch [8/10] | Train Loss: 0.2319
DenseNet121 | Epoch [9/10] | Train Loss: 0.1640
DenseNet121 | Epoch [10/10] | Train Loss: 0.1470

Final DenseNet121 training completed.


In [ ]:
# ============================================================
#  TEST PREDICTIONS FROM FINAL BASE MODELS
# ============================================================

print("\n" + "=" * 70)
print("GENERATING TEST PREDICTIONS")
print("=" * 70)

# ------------------------------------------------------------
# ResNet18
# ------------------------------------------------------------

test_pred_resnet18, test_true = predict_model(final_resnet18,test_loader)

# ------------------------------------------------------------
# ResNet34
# ------------------------------------------------------------

test_pred_resnet34, _ = predict_model(final_resnet34,test_loader)

# ------------------------------------------------------------
# DenseNet121
# ------------------------------------------------------------

test_pred_densenet121, _ = predict_model(final_densenet121,test_loader)

print("\nPrediction shapes:")

print(
    "ResNet18    :",
    test_pred_resnet18.shape
)

print(
    "ResNet34    :",
    test_pred_resnet34.shape
)

print(
    "DenseNet121 :",
    test_pred_densenet121.shape
)


GENERATING TEST PREDICTIONS

Prediction shapes:
ResNet18    : (18,)
ResNet34    : (18,)
DenseNet121 : (18,)


In [28]:
# ============================================================
# CREATE TEST META FEATURES
# ============================================================

# ------------------------------------------------------------
# Combine the three CNN probabilities
# ------------------------------------------------------------

test_meta_features = np.column_stack([test_pred_resnet18,test_pred_resnet34,test_pred_densenet121])

print("\n" + "=" * 70)
print("TEST META FEATURES")
print("=" * 70)

print(
    "Shape:",
    test_meta_features.shape
)

print("\nFirst 10 test samples and their 3 meta-features:")
print(test_meta_features[:10])


TEST META FEATURES
Shape: (18, 3)

First 10 test samples and their 3 meta-features:
[[0.12115096 0.5958656  0.14774589]
 [0.29634175 0.968567   0.6790181 ]
 [0.76489145 0.1447042  0.06784333]
 [0.08785647 0.07164308 0.32574165]
 [0.9051844  0.95264024 0.72071755]
 [0.5407794  0.55708534 0.3009864 ]
 [0.01628011 0.16422084 0.05784103]
 [0.6397513  0.23432195 0.39587986]
 [0.01173719 0.08829685 0.01930226]
 [0.00507263 0.5771954  0.16184817]]


In [22]:
# ============================================================
# CELL 20 — FINAL STACKING PREDICTION
# ============================================================

print("\n" + "=" * 70)
print("FINAL STACKING PREDICTION")
print("=" * 70)

# ------------------------------------------------------------
# Meta-learner prediction
# ------------------------------------------------------------

final_probabilities = (
    meta_learner
    .predict_proba(test_meta_features)[:, 1]
)

# ------------------------------------------------------------
# Convert probability to class
# ------------------------------------------------------------

final_predictions = (
    final_probabilities >= 0.5
).astype(int)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nFirst 10 predictions:")

for i in range(min(10, len(test_paths))):

    print(
        f"{i+1:02d}. "
        f"R18={test_pred_resnet18[i]:.4f} | "
        f"R34={test_pred_resnet34[i]:.4f} | "
        f"D121={test_pred_densenet121[i]:.4f} | "
        f"Stack={final_probabilities[i]:.4f} | "
        f"Pred={final_predictions[i]} | "
        f"True={test_labels[i]}"
    )


FINAL STACKING PREDICTION

First 10 predictions:
01. R18=0.1212 | R34=0.5959 | D121=0.1477 | Stack=0.3957 | Pred=0 | True=0
02. R18=0.2963 | R34=0.9686 | D121=0.6790 | Stack=0.5122 | Pred=1 | True=0
03. R18=0.7649 | R34=0.1447 | D121=0.0678 | Stack=0.4915 | Pred=0 | True=0
04. R18=0.0879 | R34=0.0716 | D121=0.3257 | Stack=0.2921 | Pred=0 | True=0
05. R18=0.9052 | R34=0.9526 | D121=0.7207 | Stack=0.6762 | Pred=1 | True=0
06. R18=0.5408 | R34=0.5571 | D121=0.3010 | Stack=0.5057 | Pred=1 | True=1
07. R18=0.0163 | R34=0.1642 | D121=0.0578 | Stack=0.2928 | Pred=0 | True=0
08. R18=0.6398 | R34=0.2343 | D121=0.3959 | Stack=0.4693 | Pred=0 | True=1
09. R18=0.0117 | R34=0.0883 | D121=0.0193 | Stack=0.2798 | Pred=0 | True=0
10. R18=0.0051 | R34=0.5772 | D121=0.1618 | Stack=0.3606 | Pred=0 | True=0


In [23]:
# ============================================================
# CELL 22 — STACKING EVALUATION
# ============================================================

# Compute overall classification accuracy (correct predictions / total)
accuracy = accuracy_score(test_labels,final_predictions)

# Compute precision score: TP / (TP + FP)
precision = precision_score(test_labels,final_predictions,zero_division=0)

# Compute recall score: TP / (TP + FN)
recall = recall_score(test_labels,final_predictions,zero_division=0)

# Compute F1 score: harmonic mean of precision and recall
f1 = f1_score(test_labels,final_predictions,zero_division=0)

# Compute Area Under the Receiver Operating Characteristic curve (ROC-AUC)
auc = roc_auc_score(test_labels,final_probabilities)

# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

# Compute confusion matrix detailing TN, FP, FN, and TP counts
cm = confusion_matrix(test_labels,final_predictions)

tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp)

sensitivity = tp / (tp + fn)

print("\n")
print("=" * 60)
print("STACKING ENSEMBLE RESULTS")
print("=" * 60)

print(f"Accuracy     : {accuracy:.4f}")
print(f"Sensitivity  : {sensitivity:.4f}")
print(f"Specificity  : {specificity:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"AUC          : {auc:.4f}")



STACKING ENSEMBLE RESULTS
Accuracy     : 0.6111
Sensitivity  : 0.5000
Specificity  : 0.7000
Precision    : 0.5714
F1 Score     : 0.5333
AUC          : 0.6125


In [24]:
# ============================================================
#  CLASSIFICATION REPORT
# ============================================================

print(classification_report(test_labels,final_predictions,target_names=["Normal", "Glaucoma"],zero_division=0))

              precision    recall  f1-score   support

      Normal       0.64      0.70      0.67        10
    Glaucoma       0.57      0.50      0.53         8

    accuracy                           0.61        18
   macro avg       0.60      0.60      0.60        18
weighted avg       0.61      0.61      0.61        18



In [25]:
# ============================================================
#  INDIVIDUAL CNN PERFORMANCE
# ============================================================

def calculate_metrics(
    y_true,
    probabilities
):

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    # Compute confusion matrix detailing TN, FP, FN, and TP counts
    cm = confusion_matrix(
        y_true,
        predictions
    )

    tn, fp, fn, tp = cm.ravel()

    # Compute overall classification accuracy (correct predictions / total)
    accuracy = accuracy_score(
        y_true,
        predictions
    )

    # Compute precision score: TP / (TP + FP)
    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    sensitivity = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    specificity = tn / (tn + fp)

    # Compute F1 score: harmonic mean of precision and recall
    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    # Compute Area Under the Receiver Operating Characteristic curve (ROC-AUC)
    auc = roc_auc_score(
        y_true,
        probabilities
    )

    return [
        accuracy,
        sensitivity,
        specificity,
        precision,
        f1,
        auc
    ]


resnet18_metrics = calculate_metrics(
    test_labels,
    test_pred_resnet18
)

resnet34_metrics = calculate_metrics(
    test_labels,
    test_pred_resnet34
)

densenet_metrics = calculate_metrics(
    test_labels,
    test_pred_densenet121
)

stacking_metrics = [
    accuracy,
    sensitivity,
    specificity,
    precision,
    f1,
    auc
]

results_df = pd.DataFrame(
    [
        resnet18_metrics,
        resnet34_metrics,
        densenet_metrics,
        stacking_metrics
    ],
    columns=[
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1",
        "AUC"
    ],
    index=[
        "ResNet18",
        "ResNet34",
        "DenseNet121",
        "Stacking"
    ]
)

display(
    results_df.round(4)
)

,Accuracy,Sensitivity,Specificity,Precision,F1,AUC
ResNet18,0.6111,0.500,0.7,0.5714,0.5333,0.6500
ResNet34,0.5000,0.625,0.4,0.4545,0.5263,0.5750
DenseNet121,0.6111,0.375,0.8,0.6000,0.4615,0.7125
Stacking,0.6111,0.500,0.7,0.5714,0.5333,0.6125


In [26]:
# ============================================================
# SAVE RESULTS
# ============================================================

output_dir = os.path.join(
    path2data,
    "DRION_Stacking_Results"
)

os.makedirs(
    output_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# Save models
# ------------------------------------------------------------

# Serialize and save model weights state dictionary checkpoint to disk
torch.save(
    final_resnet18.state_dict(),
    os.path.join(
        output_dir,
        "ResNet18_final.pth"
    )
)

# Serialize and save model weights state dictionary checkpoint to disk
torch.save(
    final_resnet34.state_dict(),
    os.path.join(
        output_dir,
        "ResNet34_final.pth"
    )
)

# Serialize and save model weights state dictionary checkpoint to disk
torch.save(
    final_densenet121.state_dict(),
    os.path.join(
        output_dir,
        "DenseNet121_final.pth"
    )
)

# ------------------------------------------------------------
# Save meta learner
# ------------------------------------------------------------

import joblib

joblib.dump(
    meta_learner,
    os.path.join(
        output_dir,
        "LogisticRegression_MetaLearner.pkl"
    )
)

# ------------------------------------------------------------
# Save performance table
# ------------------------------------------------------------

results_df.to_excel(
    os.path.join(
        output_dir,
        "Stacking_Performance.xlsx"
    )
)

print(
    "Models and results saved to:"
)

print(output_dir)

Models and results saved to:
F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/DRION_Stacking_Results


In [27]:
# ============================================================
#  SAVE TEST PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "Image": test_paths,

    "True_Label": test_labels,

    "ResNet18_Probability":
        test_pred_resnet18,

    "ResNet34_Probability":
        test_pred_resnet34,

    "DenseNet121_Probability":
        test_pred_densenet121,

    "Stacking_Probability":
        final_probabilities,

    "Stacking_Prediction":
        final_predictions

})

prediction_file = os.path.join(
    output_dir,
    "Stacking_Test_Predictions.xlsx"
)

prediction_df.to_excel(
    prediction_file,
    index=False
)

print(
    "Prediction file saved:"
)

print(prediction_file)

Prediction file saved:
F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/DRION_Stacking_Results\Stacking_Test_Predictions.xlsx
